# Lab 1 at home · Two tables of your own in DuckDB

**Due:** Sunday, October 25 at 11:59 PM · **Submit:** your repo URL on CourseWorks

In class you loaded one table (penguins), found a problem and fixed it. Now do the same with **two related tables you choose**. In Session 2 we'll join them, so pick two that can be connected.

You will:
1. Pick two related public datasets and upload them
2. Load both into DuckDB and check what DuckDB decided about each column
3. Find the column the two tables share (the **key**) and check how well they match
4. Write five SQL queries that help explain your data
5. Write observations and a dataset card for each table

## 1. Pick two related tables

**Related** means they share a column you could match rows on: an ID, a name, a ZIP code, a country code, a date.

**Example pairs**
- **World Bank:** any indicator download (e.g. GDP per country per year) comes with a second file of country details (region, income group). The shared column is the country code.
- **Our World in Data:** two charts for the same countries, e.g. life expectancy and health spending. Shared columns: country and year.
- **NYC Open Data:** 311 complaints and population by ZIP code, or the street tree census and neighborhood boundaries. Shared column: ZIP code or neighborhood.
- **Your own idea:** any two public tables that share a column.

**Rules**
- **Public, open data only.** Your repo is public, and later this data goes to a free AI tier. Nothing from work, nothing personal, nothing behind a login.
- **CSV format.** If a file is Excel, open it and save it as CSV first.
- **Each file under 100 MB.** Between 1,000 and 1,000,000 rows is a good size for the main table; the second can be small.
- **Reuse allowed.** Check each license or terms page, and note it in your dataset card.

**Places to look:** [NYC Open Data](https://opendata.cityofnewyork.us) · [data.gov](https://data.gov) · [World Bank Open Data](https://data.worldbank.org) · [Our World in Data](https://ourworldindata.org) · [Kaggle Datasets](https://www.kaggle.com/datasets) (check each license)

## 2. Upload them

1. Download both CSVs to your computer.
2. In this Codespace, drag both files into the **`data`** folder in the file list on the left.
3. Rename each to something short with no spaces, e.g. `gdp.csv` and `countries.csv` (right-click → **Rename**).
4. Fill in the cell below and run it. Table names should be short, lowercase, no spaces: you'll type them in your SQL.

In [3]:
FILE_A  = "ems_2023_10.csv"     # <- your main table's file name
TABLE_A = "ems"                   # <- a short name for it, e.g. "gdp"

FILE_B  = "acs_income_2023.csv"   # <- the related table's file name
TABLE_B = "acs"                 # <- a short name for it, e.g. "countries"

In [4]:
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_A = ROOT / "data" / FILE_A
DATA_B = ROOT / "data" / FILE_B

missing = [p.name for p in (DATA_A, DATA_B) if not p.exists()]
if missing:
    print("Can't find:", ", ".join(missing))
    print("Files in data/ right now:")
    for f in sorted((ROOT / "data").iterdir()):
        print("  ", f.name)
else:
    con = duckdb.connect(str(ROOT / "lab1_home.duckdb"))
    for p in (DATA_A, DATA_B):
        print(f"Found data/{p.name} ({p.stat().st_size / 1_000_000:.1f} MB)")
    print("Connected.")

Found data/ems_2023_10.csv (41.6 MB)
Found data/acs_income_2023.csv (0.0 MB)
Connected.


## 3. First look

Preview the first 10 rows of each raw file before loading.

In [8]:
con.sql(f"SELECT * FROM read_csv('{DATA_A}') LIMIT 10")

┌─────────────┬─────────────────────┬───────────────────┬─────────────────────────────┬─────────────────┬───────────────────────────┬───────────────────────────┬────────────────────────────────┬──────────────────────────────┬───────────────────────────┬─────────────────────────┬────────────────────────────────┬──────────────────────────────┬───────────────────────────────┬────────────────────────┬─────────────────────────────┬─────────────────────────┬────────────────┬───────────────────────────┬──────────────────────────┬────────────────────────┬─────────┬────────────────┬─────────────────────┬───────────────────┬─────────────────────────┬───────────────────────┬──────────────────┬─────────────────────────┬───────────────────┬────────────────────┐
│ incident_id │  incident_datetime  │ initial_call_type │ initial_severity_level_code │ final_call_type │ final_severity_level_code │ first_assignment_datetime │ valid_dispatch_rspns_time_indc │ dispatch_response_seconds_qy │ first_activatio

In [9]:
con.sql(f"SELECT * FROM read_csv('{DATA_B}') LIMIT 10")

┌───────┬────────────────┬─────────────┬─────────────┐
│ zcta  │     GEO_ID     │ B19013_E001 │ B19013_M001 │
│ int64 │    varchar     │    int64    │    int64    │
├───────┼────────────────┼─────────────┼─────────────┤
│ 10001 │ 860Z200US10001 │      123393 │       22852 │
│ 10002 │ 860Z200US10002 │       46525 │        5561 │
│ 10003 │ 860Z200US10003 │      153750 │       11191 │
│ 10004 │ 860Z200US10004 │      220592 │       53147 │
│ 10005 │ 860Z200US10005 │      211810 │       29291 │
│ 10006 │ 860Z200US10006 │      209972 │       43452 │
│ 10007 │ 860Z200US10007 │      250001 │  -333333333 │
│ 10009 │ 860Z200US10009 │       87963 │        9219 │
│ 10010 │ 860Z200US10010 │      156127 │       11635 │
│ 10011 │ 860Z200US10011 │      146571 │       13827 │
└───────┴────────────────┴─────────────┴─────────────┘
  10 rows                                  4 columns

## 4. Load both and check the types

Load each file into a table, then `DESCRIBE` it. Compare each column's type with what it *should* be. A number or date column stored as `VARCHAR` (text) is the same kind of problem you fixed in class.

In [7]:
for table, path in ((TABLE_A, DATA_A), (TABLE_B, DATA_B)):
    con.sql(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM read_csv('{path}')")
    rows = con.sql(f"SELECT count(*) FROM {table}").fetchone()[0]
    print(f"{table}: {rows:,} rows loaded")

ems: 141,460 rows loaded
acs: 206 rows loaded


In [6]:
con.sql(f"DESCRIBE {TABLE_A}")

┌────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name           │ column_type │  null   │   key   │ default │  extra  │
│            varchar             │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ incident_id                    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ incident_datetime              │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ initial_call_type              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ initial_severity_level_code    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ final_call_type                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ final_severity_level_code      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ first_assignment_datetime      │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ valid_dispatch_rspn

In [5]:
con.sql(f"DESCRIBE {TABLE_B}")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ zcta        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ GEO_ID      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ B19013_E001 │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ B19013_M001 │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

`SUMMARIZE` profiles every column at once: min, max, how many distinct values, and `null_percentage` (how much is missing).

In [10]:
con.sql(f"SUMMARIZE {TABLE_A}")

┌────────────────────────────────┬─────────────┬─────────────────────┬─────────────────────┬───────────────┬────────────────────────────┬────────────────────┬────────────────────────────┬────────────────────────────┬────────────────────────────┬────────┬─────────────────┐
│          column_name           │ column_type │         min         │         max         │ approx_unique │            avg             │        std         │            q25             │            q50             │            q75             │ count  │ null_percentage │
│            varchar             │   varchar   │       varchar       │       varchar       │     int64     │          varchar           │      varchar       │          varchar           │          varchar           │          varchar           │ int64  │  decimal(9,2)   │
├────────────────────────────────┼─────────────┼─────────────────────┼─────────────────────┼───────────────┼────────────────────────────┼────────────────────┼───────────────────────

In [11]:
con.sql(f"SUMMARIZE {TABLE_B}")

┌─────────────┬─────────────┬────────────────┬────────────────┬───────────────┬─────────────────────┬────────────────────┬─────────┬─────────┬─────────┬───────┬─────────────────┐
│ column_name │ column_type │      min       │      max       │ approx_unique │         avg         │        std         │   q25   │   q50   │   q75   │ count │ null_percentage │
│   varchar   │   varchar   │    varchar     │    varchar     │     int64     │       varchar       │      varchar       │ varchar │ varchar │ varchar │ int64 │  decimal(9,2)   │
├─────────────┼─────────────┼────────────────┼────────────────┼───────────────┼─────────────────────┼────────────────────┼─────────┼─────────┼─────────┼───────┼─────────────────┤
│ zcta        │ BIGINT      │ 10001          │ 11697          │           210 │ 10754.601941747573  │ 579.5038220763438  │ 10160   │ 10739   │ 11356   │   206 │            0.00 │
│ GEO_ID      │ VARCHAR     │ 860Z200US10001 │ 860Z200US11697 │           218 │ NULL                │ NUL

## 5. Fix what loaded wrong (if anything)

If every type looks right, write "No fixes needed" in the text cell after the next one and skip ahead.

Otherwise, open the CSV (double-click it in the `data` folder) and find **how missing values are written** in the columns that came in as text. Edit the `nullstr` list below to match, and run it for the table that needs fixing.

In [12]:
# Change TABLE_A / DATA_A to TABLE_B / DATA_B if it's the second table that needs fixing
con.sql(f"""
    CREATE OR REPLACE TABLE {TABLE_B} AS
    SELECT * FROM read_csv('{DATA_B}', nullstr = ['NA', 'N/A', ''])
""")
con.sql(f"DESCRIBE {TABLE_B}")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ zcta        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ GEO_ID      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ B19013_E001 │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ B19013_M001 │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

**What did you fix, and how did you know?** (Double-click to edit.)

I converted the special codes -666666666, -222222222, and -333333333 to NULL using nullstr. 

I identified the issue because SUMMARIZE showed extremely negative averages for income and its margin of error, despite reporting no missing values. 



## 6. Find the key: how do the two tables connect?

The **key** is the column both tables share. Type its name in each table below. It may be spelled differently in each, e.g. `Country Code` in one and `iso3` in the other.

The next cell counts the distinct key values in each table and how many of them appear in both. (It treats both as text, so a ZIP code stored as a number still matches one stored as text.)

In [13]:
KEY_A = "zipcode"   # <- the shared column's name in your main table
KEY_B = "zcta"   # <- its name in the second table

In [14]:
con.sql(f"""
    WITH a AS (SELECT DISTINCT CAST("{KEY_A}" AS VARCHAR) AS k FROM {TABLE_A}),
         b AS (SELECT DISTINCT CAST("{KEY_B}" AS VARCHAR) AS k FROM {TABLE_B})
    SELECT
        (SELECT count(*) FROM a)                           AS keys_in_a,
        (SELECT count(*) FROM b)                           AS keys_in_b,
        (SELECT count(*) FROM a WHERE k IN (SELECT k FROM b)) AS keys_in_both
""")

┌───────────┬───────────┬──────────────┐
│ keys_in_a │ keys_in_b │ keys_in_both │
│   int64   │   int64   │    int64     │
├───────────┼───────────┼──────────────┤
│       226 │       206 │          206 │
└───────────┴───────────┴──────────────┘

**Describe the relationship** (double-click to edit):

- **Key column:** in the main table it's called zipcode, in the second zcta
- **How they relate:** does one row in the second table match *one* row in the main table, or *many*? (e.g. one country has many GDP rows, one per year)

This is a many-to-one relationship: many EMS incidents can match one income record through the same geographic code.

- **How well they match:** how many keys appear in only one table? Give one example and a guess at why (spelling, codes vs. names, regions mixed in with countries...)

The EMS table contains 225 distinct non-null ZIP codes, of which 206 match the income table and 19 do not. For example, 83 is not a valid five-digit ZIP code and may reflect a recording error. The query reports 226 keys in the EMS table because it also counts NULL as one distinct row. All 206 income-table codes match because this table was filtered to codes appearing in the EMS extract.



## 7. Five queries that explain your data

Write one query of each kind. Under each, double-click the text cell and write one line: **what does this result tell us?** Use either table.

Tip: if a column name has spaces or capitals, wrap it in double quotes: `"Country Name"`. Start from a query in the in-class notebook and change one thing.

### Query 1 · Size and shape
*How many rows, and over what time period or area?* Useful SQL: `count(*), min(), max()`

In [18]:
con.sql(f"""
    SELECT count(*) AS total_incidents,
           min(incident_datetime) AS first_incident,
           max(incident_datetime) AS last_incident,
           count(DISTINCT zipcode) AS unique_zipcodes
    FROM {TABLE_A}
""")

┌─────────────────┬─────────────────────┬─────────────────────┬─────────────────┐
│ total_incidents │   first_incident    │    last_incident    │ unique_zipcodes │
│      int64      │      timestamp      │      timestamp      │      int64      │
├─────────────────┼─────────────────────┼─────────────────────┼─────────────────┤
│          141460 │ 2023-10-01 00:00:17 │ 2023-10-31 23:59:55 │             225 │
└─────────────────┴─────────────────────┴─────────────────────┴─────────────────┘

**What this tells us: The dataset contains 141,460 EMS incidents from October 1 to October 31, 2023, with 225 distinct non-null ZIP codes.** 

### Query 2 · Filter
*Which rows meet a condition you care about?* Useful SQL: `WHERE`

In [ ]:
con.sql(f"""
    SELECT count(*) AS incidents_over_15_min,
           round(avg(incident_response_seconds_qy) / 60.0, 2) AS avg_response_minutes
    FROM {TABLE_A}
    WHERE valid_incident_rspns_time_indc = 'Y'
      AND incident_response_seconds_qy > 900
""")

┌───────────────────────┬──────────────────────┐
│ incidents_over_15_min │ avg_response_minutes │
│         int64         │        double        │
├───────────────────────┼──────────────────────┤
│                 29881 │                34.37 │
└───────────────────────┴──────────────────────┘

**What this tells us: Among incidents flagged as having valid response times, 29,881 exceeded 15 minutes. Their average response time was 34.37 minutes.** 

### Query 3 · Group
*How does a count or average differ across categories?* Useful SQL: `GROUP BY`

In [20]:
con.sql(f"""
    SELECT borough, count(*) AS valid_incidents,
           round(avg(incident_response_seconds_qy) / 60.0, 2) AS avg_response_minutes
    FROM {TABLE_A}
    WHERE valid_incident_rspns_time_indc = 'Y'
    GROUP BY borough
    ORDER BY avg_response_minutes DESC
""")

┌──────────────────────────┬─────────────────┬──────────────────────┐
│         borough          │ valid_incidents │ avg_response_minutes │
│         varchar          │      int64      │        double        │
├──────────────────────────┼─────────────────┼──────────────────────┤
│ MANHATTAN                │           32199 │                15.91 │
│ BRONX                    │           31371 │                15.67 │
│ BROOKLYN                 │           36519 │                12.54 │
│ QUEENS                   │           27873 │                 11.0 │
│ RICHMOND / STATEN ISLAND │            5793 │                 9.21 │
│ UNKNOWN                  │               2 │                 2.39 │
└──────────────────────────┴─────────────────┴──────────────────────┘

**What this tells us: Among the five named boroughs, Manhattan had the longest average valid response time at 15.91 minutes, while Staten Island had the shortest at 9.21 minutes. These comparisons do not adjust for incident severity or other differences.** 

### Query 4 · Rank
*What are the top or bottom 10 of something?* Useful SQL: `ORDER BY ... LIMIT 10`

In [22]:
con.sql(f"""
    SELECT zipcode, count(*) AS incident_count
    FROM {TABLE_A}
    WHERE zipcode IS NOT NULL
    GROUP BY zipcode
    ORDER BY incident_count DESC, zipcode
    LIMIT 10
""")

┌─────────┬────────────────┐
│ zipcode │ incident_count │
│  int64  │     int64      │
├─────────┼────────────────┤
│   10457 │           2347 │
│   10456 │           2340 │
│   11212 │           2308 │
│   10467 │           2238 │
│   11207 │           2207 │
│   10453 │           2073 │
│   10029 │           2030 │
│   10451 │           1953 │
│   11201 │           1884 │
│   10458 │           1864 │
└─────────┴────────────────┘
  10 rows        2 columns

**What this tells us: ZIP code 10457 had the most recorded incidents, with 2,347, followed by 10456 with 2,340. These are incident counts, not population-adjusted rates.** 

### Query 5 · Quality check
*What's missing, duplicated or impossible?* Useful SQL: `count(*) - count(column), count(DISTINCT ...)`

In [23]:
con.sql(f"""
    SELECT count(*) AS total_rows,
           count(*) - count(DISTINCT zcta) AS duplicate_or_missing_zcta_rows,
           count(*) - count(B19013_E001) AS missing_income,
           count(*) - count(B19013_M001) AS missing_margin_of_error,
           count(*) FILTER (WHERE B19013_E001 < 0) AS negative_income_values
    FROM {TABLE_B}
""")

┌────────────┬────────────────────────────────┬────────────────┬─────────────────────────┬────────────────────────┐
│ total_rows │ duplicate_or_missing_zcta_rows │ missing_income │ missing_margin_of_error │ negative_income_values │
│   int64    │             int64              │     int64      │          int64          │         int64          │
├────────────┼────────────────────────────────┼────────────────┼─────────────────────────┼────────────────────────┤
│        206 │                              0 │              0 │                       0 │                     23 │
└────────────┴────────────────────────────────┴────────────────┴─────────────────────────┴────────────────────────┘

**What this tells us: After cleaning, the income table retains 206 rows with no duplicate or missing ZCTA codes. Income is missing for 23 areas and the margin of error for 26 areas; no negative income values remain.** 

## 8. Three observations

Specific things you noticed that you'd want to check before trusting this data. Point to numbers.

1. The ACS table contains 23 missing income estimates out of 206 areas (11.17%) and 26 missing margins of error (12.62%) after converting special codes to NULL. These gaps should be checked before comparing income across areas.
2. Of the 225 distinct non-null EMS ZIP codes, 19 do not match the ACS table. One example is 83, which is not a five-digit ZIP code. Unmatched codes need investigation, and ZIP codes and ZCTAs should not be assumed to represent identical boundaries.
3. Among incidents with valid response times, Manhattan’s average was 15.91 minutes, compared with 9.21 minutes in Staten Island. Before interpreting this difference as evidence of inequality, I would check incident severity, call types, and whether the pattern persists beyond this one-month sample.

**One question you'd want a non-technical person to be able to ask of this data:**

Do lower-income areas in New York City experience longer EMS response times, and are the differences greater in dispatch time or travel time?



## 9. Dataset cards

| | Main table | Second table |
|---|---|---|
| **Dataset name** | EMS Incident Dispatch Data | 2023 ACS 5-Year Estimates, Table B19013: Median Household Income |
| **Source (link)** | https://data.cityofnewyork.us/Public-Safety/EMS-Incident-Dispatch-Data/76xm-jjuj/about_data| https://www2.census.gov/programs-surveys/acs/summary_file/2023/table-based-SF/data/5YRData/acsdt5y2023-b19013.dat|
| **Who collected it, and why** | The Fire Department of the City of New York (FDNY) records EMS incidents through its dispatch system to manage and document emergency responses.| The U.S. Census Bureau collects the American Community Survey to describe population, housing, and socioeconomic conditions and support planning and public policy.|
| **License or terms of use** | Publicly reusable under NYC Open Data policies and terms. Data is provided without warranties of accuracy or completeness.| Public domain / CC0 1.0, as listed in the official Data.gov catalog.|
| **Rows × columns** | 141,460 rows × 31 columns| 206 rows × 4 columns|
| **Time period covered** | October 1–31, 2023| 2019–2023. Income estimates are expressed in 2023 inflation-adjusted dollars.|
| **How often it's updated** | Annually, according to the source metadata. This CSV is a fixed October 2023 extract.| New ACS five-year estimates are released annually. This CSV uses the 2023 release.|

## 10. Submit

1. Run the cell below to close the connection.
2. Fill in the lifecycle map of your data in `labs/lab1.md`.
3. **Save this notebook (Cmd+S)**, then commit and push: **Source Control** → message `Lab 1` → **Commit** → **Sync Changes**.
4. Check on GitHub that both CSVs, both notebooks and `labs/lab1.md` are there, with results visible.
5. Paste your repo URL into the Lab 1 assignment on CourseWorks.

In [24]:
con.close()
print("Done. Save, commit, push, and submit your repo URL.")

Done. Save, commit, push, and submit your repo URL.
